# Uncertainty-aware planning для Cosmos Policy

**Постановка задачи, гипотезы, эксперименты, результаты и обзор литературы**  
Актуальность: 20 августа 2026 года.

Цель исследования — понять, можно ли заранее обнаруживать ненадёжные предсказания Cosmos Policy и использовать эту информацию при выборе action chunk в planning. Основная среда — LIBERO-PRO с визуальными, объектными, пространственными и языковыми OOD-вариациями.

Ноутбук разделён на две части:

1. **Собственное исследование:** задача, модель, гипотезы, формулы, протокол, результаты и следующие эксперименты.
2. **Обзор литературы:** статьи из проекта, близкие современные методы и позиционирование относительно SOTA.

Новые серверные результаты находятся в [experiments/LIBERO_PHASE1_RESULTS.ipynb](experiments/LIBERO_PHASE1_RESULTS.ipynb); исторические исполняемые rollout, таблицы, графики и видео находятся в [ysda_world_models.ipynb](ysda_world_models.ipynb). Этот notebook содержит устойчивое описание исследования и не дублирует тяжёлые экспериментальные outputs.

## 1. Задача исследования

Cosmos Policy может семплировать несколько action candidates и выбирать candidate с наибольшим predicted value. В сложных и OOD-сценах этого недостаточно: value может быть плохо откалиброван, а высокое значение — соответствовать физически ненадёжному захвату, переносу или размещению объекта.

Мы исследуем четыре связанных вопроса:

1. **Failure prediction:** можно ли по доступным до исполнения chunk сигналам предсказать будущий fail?
2. **World-model reliability:** связана ли uncertainty с последующим расхождением predicted future и реального observation?
3. **Candidate ranking:** можно ли ранжировать несколько action chunks лучше, чем по одному `max(value)`?
4. **Epistemic uncertainty:** можно ли перейти от эвристик одного checkpoint к disagreement независимо обученных flow-моделей?

Рабочая цель planning формулируется как

$$
a^*=\arg\max_{a_i}
\left[\text{expected task progress}(a_i)-\text{risk}(a_i)\right].
$$

Risk должен быть доступен **до выполнения** candidate. Ошибки, вычисленные после `env.step`, используются для анализа и обучения proxy-модели, но не являются online planning signal.

## 2. Модель, входы и единица принятия решения

На каждом policy query модель получает:

- текущую third-person RGB-картинку `agentview_image`;
- текущую wrist RGB-картинку `robot0_eye_in_hand_image`;
- proprio: положение gripper, end-effector position и quaternion;
- языковую команду в виде cached T5 embedding.

Для LIBERO joint latent sequence имеет структуру

```text
[blank, current proprio, current wrist image, current primary image,
 action chunk, future proprio, future wrist image, future primary image, value]
```

Action chunk содержит 16 семимерных управляющих векторов:

$$
A_i=(a_{i,0},\ldots,a_{i,15})\in\mathbb R^{16\times 7}.
$$

В историческом baseline выбранный chunk выполнялся open-loop до 16 simulator steps. В новом overlap-эксперименте prediction horizon остаётся $H=16$, но выполняются только первые $K=8$ действий. После этого новый query получает **реальные** изображения и proprio из LIBERO, а не предсказанные моделью кадры. Поэтому `query=3` означает четвёртый вызов модели перед выполнением очередного chunk, а observation после выполненного prefix уже содержит последствия выбранных действий.

Основные точки входа:

- `cosmos_policy/experiments/robot/cosmos_utils.py`: preprocessing, `get_action`, decoding и unnormalization;
- `cosmos_policy/experiments/robot/libero/uncertainty_comparison.py`: paired rollout, метрики и planning selection;
- `cosmos_policy/experiments/robot/libero/uncertainty_metrics.py`: output-level uncertainty;
- `cosmos_policy/experiments/robot/libero/vfd_metrics.py`: denoising trace, path dispersion и exact block-wise VFD.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

nodes = [
    ('Real LIBERO obs', 'agentview + wrist RGB\n9D proprio'),
    ('Preprocessing', 'flip / JPEG / resize\nproprio normalization'),
    ('Conditioning', 'T5 command\nlatent masks'),
    ('Cosmos Policy', 'joint denoising\n5 action NFE'),
    ('Candidate', '16 x 7 actions\nfuture + value'),
    ('Risk-aware ranker', 'value\nuncertainty / risk'),
    ('env.step', 'execute K < H\nopen-loop actions'),
    ('Next real obs', 'new query\nclosed-loop update'),
]

fig, ax = plt.subplots(figsize=(18, 4.8))
ax.set_axis_off()
x_positions = [0.01, 0.135, 0.26, 0.385, 0.51, 0.635, 0.76, 0.885]
y, w, h = 0.55, 0.105, 0.34

for index, ((title, body), x) in enumerate(zip(nodes, x_positions)):
    color = '#e8f3ff' if index in {3, 5} else '#f7f7f2'
    box = FancyBboxPatch(
        (x, y - h / 2), w, h,
        boxstyle='round,pad=0.012,rounding_size=0.015',
        linewidth=1.4, facecolor=color, edgecolor='#333333',
    )
    ax.add_patch(box)
    ax.text(x + w / 2, y + 0.075, title, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(x + w / 2, y - 0.055, body, ha='center', va='center', fontsize=8)
    if index < len(nodes) - 1:
        ax.add_patch(FancyArrowPatch(
            (x + w + 0.004, y), (x_positions[index + 1] - 0.004, y),
            arrowstyle='-|>', mutation_scale=13, linewidth=1.2, color='#444444',
        ))

ax.text(
    0.5, 0.12,
    'Planning выбирает candidate до исполнения; prediction error становится известна только после chunk.',
    ha='center', va='center', fontsize=11,
)
plt.show()

## 3. Проверяемые гипотезы

| ID | Гипотеза | Наблюдаемое следствие |
|---|---|---|
| H1 | Перед физическим fail предсказания становятся менее устойчивыми | Action/value/future uncertainty возрастает до failure onset |
| H2 | Часть fail проявляется как brittle low-disagreement режим, а не просто высокая дисперсия | Композитный score из низкой dispersion и mean value лучше предсказывает fail, чем raw std |
| H3 | Future uncertainty отражает ошибку world model | Сигнал до chunk коррелирует с последующим proprio/object/semantic prediction error |
| H4 | Risk-aware ranking лучше `max(value)` | На одинаковых task/init/rollout seeds повышается success rate и снижается paired regret |
| H5 | Разные stochastic seeds одного checkpoint недостаточны для epistemic uncertainty | Output/path dispersion переносится хуже, чем cross-model disagreement независимых LoRA members |
| H6 | Semantic future error важнее pixel reconstruction error | DINO/V-JEPA/object-state error лучше связан с fail и task progress, чем RGB MSE |
| H7 | Старый неисполненный tail и новый aligned prefix расходятся перед локальным fail | TIDE/STAC-style overlap score растёт до drop/contact event и улучшает requery или ranking |

H1–H4 исследованы в пилотных rollout. H5 получил корректную instrumentation, но требует обучения ensemble. H6 остаётся следующим representation experiment. H7 сформулирована 20 августа 2026 года; результаты по ней ещё не получены.

## 4. Использованные метрики

Для одного observation генерируются stochastic candidates

$$
c_i=(A_i,\hat s_i^{future},V_i),\qquad i=1,\ldots,N,
$$

с разными sampling seeds одного checkpoint.

### 4.1. Разброс между stochastic candidates

$$
U_{action}^{seed}=\operatorname{mean}_{t,d}
\operatorname{std}_i A_i[t,d],
$$

$$
U_{value}^{seed}=\operatorname{std}_i V_i,
\qquad
R_{value}^{seed}=\max_i V_i-\min_i V_i.
$$

Аналогично считаются `future_proprio_std`, pixel std будущих изображений и std соответствующих latent frames. Эти величины смешивают sampling noise, мультимодальность допустимых действий и model uncertainty.

### 4.2. Внутренняя согласованность одного latent sample

Низкоразмерный action chunk повторяется внутри action latent frame. Для копий $A_{i,k}$ вычисляется

$$
\sigma_i(t,d)=\operatorname{std}_k A_{i,k,t,d},
\qquad
U_i^{action}=\|\sigma_i(0,:)\|_2.
$$

Кодовое имя: `latent_action_first_step_copy_l2_std`. Первое действие выбрано потому, что оно исполняется сразу после query.

Для value использовалась неоднородность элементов value latent frame:

$$
U_i^{value}=\operatorname{std}(L_i^{value}).
$$

Кодовое имя: `latent_value_element_std_mean`. Обе метрики являются decoder-consistency proxies, а не независимыми posterior samples.

### 4.3. Handcrafted episode-risk scores

Для exploratory fail prediction query-метрики усреднялись до `query <= 6`, после чего стандартизовались между эпизодами анализируемой группы. Точные формулы использованных scores:

$$
R_{value}=\frac14\left[
-z(\overline{value\_std})
-z(\overline{value\_range})
-z(\overline{latent\_value\_std})
-0.5z(\overline{value\_mean})
\right],
$$

$$
R_{action}=\frac12\left[
-z(\overline{action\_first\_step\_std})
-z(\overline{latent\_action\_copy\_std})
\right],
$$

$$
R_{classic}=\frac14\left[
z(\overline{value\_std})
+z(\overline{value\_range})
+z(\overline{action\_first\_step\_std})
+z(\overline{future\_image\_std})
\right].
$$

В CSV они называются `risk_overconfidence_value`, `risk_overconfidence_action` и `risk_classic_high_uncertainty`. Название `overconfidence_value` историческое: score включает не только низкую value dispersion, но и член `-0.5 z(value_mean)`, поэтому его нельзя описывать как чистую «уверенность при завышенном value».

Эти z-score вычислялись на анализируемой группе эпизодов. Следовательно, они подходят для exploratory separation, но для deployment mean/std и threshold должны быть зафиксированы на отдельном calibration split.

### 4.4. Prediction error после выполнения chunk

После исполнения выбранного chunk сохраняются:

- RGB `MSE`, `MAE`, global `SSIM`, `PSNR` для primary и wrist images;
- proprio `L2`, отдельно end-effector position error;
- absolute value error относительно chunk/final success.

Эти метрики отвечают на вопрос, разошлось ли предсказание с реальностью. Они не могут непосредственно участвовать в выборе уже выполняемого chunk.

## 5. Проверенные planning-стратегии

Baseline:

$$
i^*_{max-value}=\arg\max_i V_i.
$$

Для risk-aware выбора value и risk нормируются среди candidates текущего query:

$$
z(x_i)=\frac{x_i-\mu_x}{\sigma_x+\varepsilon}.
$$

Проверенные score:

$$
S_i^{action}=z(V_i)-1.0\,z(U_i^{action}),
$$

$$
S_i^{value}=z(V_i)-2.0\,z(U_i^{value}),
$$

$$
S_i^{combined}=z(V_i)-\lambda
\left[\tfrac12z(U_i^{action})+\tfrac12z(U_i^{value})\right].
$$

Выбирается $i^*=\arg\max_i S_i$.

В этих экспериментах `action`, `future` и `value` кандидата генерировались **параллельно в одной joint latent sequence**. Это дешёвый candidate reranking. Полный planning из Cosmos Policy использует более дорогую цепочку `action -> несколько future states -> несколько value estimates`; поэтому наши результаты нельзя называть полным воспроизведением planning-протокола статьи.

## 6. Экспериментальный протокол

### Paired natural success/fail

Основная единица сравнения — фиксированные `suite`, `task_id` и `init_state_id`. Меняется только rollout/model seed. Запуски продолжаются, пока для одного начального состояния не собраны и success, и fail.

Это не буквальные twins: разные model seeds уже на первом query могут выбрать разные действия, после чего реальные траектории расходятся. Такой дизайн контролирует задачу и начальную сцену, но не делает состояния на поздних query одинаковыми.

### Момент ошибки

Видео и simulator state используются для определения физического failure onset: потеря grasp, падение объекта, промах контакта, неверное размещение или необратимое движение от цели. Timeout сам по себе не считается достаточным объяснением fail. Для раннего предсказания анализируются только queries, доступные до onset; для группового сравнения fail-траектории обрезаются до длины соответствующего success.

### Prediction versus reality

На query сохраняются predicted future/value и uncertainty. После chunk они сопоставляются с реальным observation. Это позволяет разделить:

- online risk до действия;
- prediction error после действия;
- итоговый episode success.

### Защита от leakage

- все seeds одного init state должны находиться в одном split;
- коэффициенты score и thresholds подбираются только на train/validation configurations;
- test tasks/init states не используются для выбора признаков;
- `prediction_error_*`, `final_t`, итоговый success и video-derived onset не входят в online feature set.

## 7. Результаты: success/fail и prediction error

### Исторические single-configuration пилоты

На `libero_spatial_with_milk/task5/init0` ранее были собраны mixed-40,
seed-70000 и seed-80000 выборки. Value-overconfidence достигал AUROC `0.905` и
`0.917` на первых двух наборах, но упал до `0.586` на следующих 24 seeds.
Action-overconfidence переносился лучше (`0.724`, `0.806`, `0.707`), однако
эти scores и normalization подбирались внутри одной конфигурации.

Этот этап показал возможность natural paired analysis, но не доказал
multi-task перенос.

### Phase 1: multi-task server validation

| Split | Эпизоды | Success | Fail | Success rate |
|---|---:|---:|---:|---:|
| ID | 72 | 72 | 0 | 100.0% |
| LIBERO-PRO OOD | 106 | 82 | 24 | 77.4% |

| Suite | Task | Init | Success | Success rate |
|---|---:|---:|---:|---:|
| `libero_10_with_milk` | 9 | 0 | 3/4 | 75.0% |
| `libero_10_with_mug` | 4 | 0 | 2/4 | 50.0% |
| `libero_goal_with_mug` | 9 | 0 | 3/4 | 75.0% |
| `libero_spatial_with_milk` | 5 | 0 | 12/24 | 50.0% |
| `libero_spatial_with_mug` | 0 | 0 | 8/12 | 66.7% |
| `libero_spatial_with_yellow_book` | 5 | 0 | 5/6 | 83.3% |
| `libero_spatial_with_yellow_book` | 8 | 0 | 3/6 | 50.0% |

Получено семь fixed `suite/task/init_state` со смесью исходов. Для strict early
анализа использовались только `query=0..3`, до самого раннего failure event:

| Early metric, mean over query 0-3 | Pooled within-group AUROC | Direction | High / tie / low groups |
|---|---:|---|---:|
| `latent_action_copy_std_mean_mean_over_samples` | 0.671 | `high=failure` | 5 / 1 / 1 |
| `value_mean` | 0.594 | `high=failure` | 3 / 1 / 3 |
| `future_image_pixel_std_mean` | 0.579 | `high=failure` | 4 / 0 / 3 |
| `action_std_mean` | 0.576 | `high=failure` | 3 / 0 / 4 |
| `value_std` | 0.537 | `high=failure` | 4 / 1 / 2 |
| `value_range` | 0.522 | `high=failure` | 4 / 1 / 2 |
| `action_first_step_l2_std` | 0.512 | `high=failure` | 2 / 0 / 5 |

Главный новый результат: internal action-latent consistency остаётся лучшим
кандидатом, но его pooled within-group AUROC равен только **0.671**. Направление
`high=failure` выполняется в 5/7 групп, одна даёт tie и одна инверсию.
`action_first_step_l2_std` находится около случайного уровня (`0.512`), а
`value_std` и `value_range` дают `0.537` и `0.522`.

После контроля `suite/task/init_state/query_idx` сильные pooled correlations с
prediction error исчезают. Максимальная содержательная связь —
internal future-proprio consistency против proprio L2, Spearman `rho=0.139`.
Значит прежние корреляции около 0.8 в основном отражали фазу эпизода.

Артефакты: [Phase 1 notebook](experiments/LIBERO_PHASE1_RESULTS.ipynb) и
[полный отчёт](experiments/campaigns/phase1_analysis_20260724/README.md).

## 8. Результаты: реальный candidate planning

Стратегии были проверены непосредственно в LIBERO-PRO: на каждом query генерировались candidates, score выбирал один chunk, и выбранный chunk исполнялся в среде.

| Стратегия | $\lambda$ | Fast12 A | Fast12 B с видео |
|---|---:|---:|---:|
| `max_value` | 0 | 5/12 (41.7%) | 6/12 (50.0%) |
| action penalty | 1 | **8/12 (66.7%)** | 6/12 (50.0%) |
| value penalty | 2 | 5/12 (41.7%) | **7/12 (58.3%)** |
| combined penalty | 2 | не запускался | 4/12 (33.3%) |

Risk-aware стратегии действительно меняли решение: action/value variants выбирали не `max(value)` примерно в 44–53% queries. Значит эксперимент проверяет реальное действие ranker, а не только post-hoc score.

Первый Fast12 поддержал H4 для action penalty: `+3` success относительно baseline. Повторный сбор этот выигрыш не воспроизвёл; value penalty дал небольшой выигрыш только во втором сборе, а combined strategy ухудшила результат. При 12 rollout разница в один эпизод равна 8.3 percentage points, поэтому ни одну стратегию пока нельзя объявить устойчиво лучшей.

Корректный вывод: **uncertainty-aware reranking перспективен и способен изменить исход отдельных rollout, но текущие proxy-метрики и гиперпараметры ещё не дают воспроизводимого превосходства над `max(value)`.**

Артефакты: [Fast12 A summary](experiments/uncertainty/planning_best2_milk_task5_init0_fast12_20260529__analysis/planning_strategy_summary.csv), [Fast12 B summary](experiments/uncertainty/planning_video4_milk_task5_init0_fast12_20260529__analysis/planning_strategy_summary.csv).

## 9. Переход к denoising uncertainty: VFD

Для Cosmos EDM state

$$
x_\sigma=x_0+\sigma\epsilon
$$

из noisy state и x0 prediction восстанавливается

$$
\hat\epsilon=\frac{x_\sigma-\hat x_0}{\sigma},
\qquad
\hat v_t=\hat\epsilon-\hat x_0,
\qquad
t=\frac{\sigma}{1+\sigma}.
$$

В координатах Cosmos вес VFD равен $\kappa=1/\sigma$. Для semantic block $g$ точный cross-model estimator:

$$
U_g^{VFD}=
\frac{1}{M(M-1)N_s}
\sum_{i\ne j}\sum_l
\frac{1}{\sigma_{i,l}}
\frac{\|v_i(x_{i,l})-v_j(x_{i,l})\|_2^2}{d_g}.
$$

Здесь модель $j$ обязательно оценивается **в той же точке** траектории $x_{i,l}$ модели $i$. Деление на размер блока $d_g$ не позволяет image latents доминировать только из-за числа элементов.

При нескольких seeds одного checkpoint доступны скорости в разных точках $v(x_{i,l})$ и $v(x_{j,l})$. Их расстояние называется в коде `flow_path_dispersion`:

$$
D_g^{path}=\mathbb E_{i<j,l}
\left[\frac{1}{\sigma_l}
\frac{\|v(x_{i,l})-v(x_{j,l})\|_2^2}{d_g}\right].
$$

Это sampling/path variability, а не epistemic cross-model VFD.

## 10. E0: реализация и sanity checks

Реализовано:

- callback после каждого solver denoiser evaluation без дополнительного forward;
- запись только блоков `action`, `future_proprio`, `future_wrist_image`, `future_image`, `value`;
- исключение terminal clean-pass;
- exact `compute_blockwise_vfd` для матрицы `(trajectory model, evaluator model, step, ...)`;
- compact `flow_path_*` aggregates в `collect` и `collect-paired` по флагу `--record-denoising-trace`;
- отсутствие сохранения больших raw latent traces на диск.

Проверка на физической GPU 2, NVIDIA H100 80 GB:

| Проверка | Результат |
|---|---|
| Unit/sanity tests | 7/7 passed |
| Action с tracing и без него | bitwise equal |
| Полный generated latent | bitwise equal |
| Solver evaluations при пяти action denoising steps | 4; final clean-pass исключён |
| Сигмы | 80.0, 42.2911, 20.9724, 9.61825 |
| Exact VFD дублированной модели | 0.0 для всех блоков |
| Path dispersion идентичных traces | 0.0 |

Для seeds 195/196 weighted path dispersion получилась почти одинаковой по всем блокам: action `0.09468`, future proprio `0.09396`, future image `0.09247`, wrist image `0.09409`, value `0.09470`. Это указывает на доминирование общего initial-noise spread и подтверждает, что разные seeds одного checkpoint нельзя выдавать за epistemic VFD.

Полный отчёт: `experiments/uncertainty/vfd_e0_smoke.json`. E0 доказал корректность и неинвазивность instrumentation; содержательный VFD-эксперимент начинается с двух независимо обученных LoRA members.

## 11. Основные выводы собственного исследования

### Что поддерживается данными

1. OOD screening работает: 72/72 ID rollout успешны, тогда как в специально
   отобранных LIBERO-PRO OOD конфигурациях получено 82/106 success.
2. Найдено семь natural mixed `suite/task/init_state`, пригодных для paired
   calibration и planning. Три из них имеют около 50% success.
3. Лучший strict-early signal — internal action-latent consistency,
   `latent_action_copy_std_mean_mean_over_samples`, AUROC `0.671` на 60
   эпизодах из семи mixed groups.
4. Risk-aware candidate selection технически меняет решение относительно
   `max(value)` и способна улучшать отдельные rollout.
5. Denoising trace реализован без изменения policy output; exact block-wise VFD
   готов к проверке после обучения независимых LoRA members.
6. Drop detector полезен как temporal event: часть траекторий после события
   восстанавливается и успешно завершает задачу.

### Что новые данные опровергли или ослабили

1. `action_first_step_l2_std` не является универсальным ранним predictor:
   multi-task AUROC равен `0.512`.
2. `value_std` и `value_range` не дают устойчивого переноса между задачами.
3. Высокие pooled correlations uncertainty/prediction-error были в основном
   confounded фазой эпизода; после контроля query остаются слабые связи.
4. Нельзя сводить natural fail к простой формуле «чем выше std, тем хуже».

### Что пока не доказано

1. Нет зафиксированного predictor, проверенного на внешнем seed/init/task
   holdout.
2. Нет воспроизводимого превосходства planning strategy над `max(value)`.
3. Output dispersion одного checkpoint не является чистой epistemic
   uncertainty и плохо калибрует world-model error.
4. PRO diagnostics не заменяют официальные constraints LIBERO-Safety.
5. Не проверен candidate ranker на клонах одного simulator state.

Текущий результат — строгий отрицательно-положительный итог: полезный ранний
latent-action signal найден, но старые более сильные single-task выводы не
перенеслись. Это хороший calibration baseline, а не готовая safety-система.

## 12. Следующие эксперименты

| Этап | Вопрос | Дизайн | Критерий результата |
|---|---|---|---|
| E1 | Даёт ли ensemble настоящий epistemic signal? | Frozen Cosmos base + 2 LoRA members, разные seeds и bootstrap standard-LIBERO demos; cross-evaluation каждой модели на обеих trajectories | Duplicate ensemble даёт 0; независимые LoRA дают ненулевой, permutation-invariant VFD |
| E2 | Предсказывает ли VFD natural fail заранее? | ID calibration и held-out LIBERO-PRO tasks/init states; анализ только до physical onset | AUROC/AUPRC, TPR@10% FPR, lead time; bootstrap CI по init states |
| E3 | Связан ли VFD с ошибкой world model? | После каждого chunk: proprio, object pose/contact и DINO/V-JEPA future error | Future-block VFD превосходит pixel MSE proxies и stochastic path dispersion |
| E4 | Ранжирует ли score разные actions? | Сохранить simulator states перед grasp/contact; из каждого state исполнить K=8 candidates в клонах среды | Pairwise ranking accuracy, NDCG, oracle regret относительно actual branch outcome |
| E5 | Улучшает ли VFD real planning? | Paired `max(value)` против VFD score/gate на нескольких OOD suites | Paired success delta, McNemar, bootstrap CI; latency и coverage-risk curve |

Предлагаемый candidate score:

$$
S_i=z(\mu_{V,i})
-\lambda_a z(U_{a,i}^{VFD})
-\lambda_f z(U_{future,i}^{VFD})
-\lambda_v z(U_{V,i}^{VFD})
-\lambda_r z(R_i^{temporal}).
$$

Если все candidates ненадёжны, нужен reject/replan gate:

$$
\min_i R_i>q_{1-\alpha}^{cal}
\Longrightarrow
\text{new observation / shorter chunk / resampling / fallback}.
$$

Главный следующий научный эксперимент — E4. Он сравнивает actions из **одного и того же simulator state** и тем самым отделяет качество candidate ranking от расхождения посещённых rollout states.

# Часть II. Обзор литературы

В области нет одного универсального SOTA: failure detection, epistemic uncertainty, test-time reranking, world-model representation и active data collection оцениваются по разным протоколам. Поэтому работы ниже сгруппированы по роли в нашей системе, а результаты свежих arXiv preprints 2026 года рассматриваются как предварительные.

## 13. Карта наиболее близких методов

| Подзадача | Наиболее близкие работы | Что переносим в проект |
|---|---|---|
| Joint world-model policy | [Cosmos Policy](https://arxiv.org/abs/2601.16163) | Joint action/future/value latent и candidate planning |
| Early failure prediction | [Foresight](https://arxiv.org/abs/2606.23085), [SAFE](https://arxiv.org/abs/2506.09937) | Temporal risk по истории current/predicted future latents, conformal threshold |
| Epistemic uncertainty flow-policy | [VFD / SAVE](https://arxiv.org/abs/2606.18043) | Cross-model velocity-field disagreement |
| Test-time action reranking | [TACO](https://arxiv.org/abs/2512.02834), [ReconVLA](https://arxiv.org/abs/2604.16677) | Support score и calibrated action-error bound |
| Cheap online probes | [FIPER](https://arxiv.org/abs/2510.09459), [ActProbe](https://arxiv.org/abs/2606.08508), [SCALE](https://arxiv.org/abs/2602.04208) | OOD, temporal chunk consistency и single-pass uncertainty |
| World-model representation | [Reconstruction or Semantics?](https://arxiv.org/abs/2605.06388) | DINO/V-JEPA semantic future error вместо одного pixel MSE |
| Online model correction | [Feedback World Model](https://arxiv.org/abs/2605.15705) | Коррекция latent dynamics по prediction residual |
| Hard evaluation | [LIBERO-Safety](https://arxiv.org/abs/2606.23686) | Physical/semantic safety и OOD failure taxonomy |

Для нашей темы наиболее важна комбинация трёх направлений: VFD как epistemic estimator, Foresight как temporal failure detector и TACO/ReconVLA как ориентиры для реального candidate selection.

## 14. Статьи из папки проекта

### Cosmos Policy

Источники: [локальный PDF](articles/2601.16163v1.pdf), [arXiv](https://arxiv.org/abs/2601.16163).

Cosmos Policy моделирует joint sequence

$$
(s_t,a_{t:t+H},\hat s_{t+H},\hat V(\hat s_{t+H})).
$$

В полном planning авторы семплируют action candidates, для каждого получают несколько future states и несколько value estimates. В описанном варианте один action оценивается по 3 future predictions и 5 value predictions на future, то есть по 15 value estimates. На сложных ALOHA-задачах world-model planning улучшил средний результат примерно на 12.5 percentage points относительно прямой политики.

### Reconstruction or Semantics?

Источники: [локальный PDF](articles/2605.06388v1.pdf), [arXiv](https://arxiv.org/abs/2605.06388).

Главный вывод: pixel reconstruction quality недостаточно для выбора world model в робототехнике. V-JEPA 2.1, DINO и SigLIP лучше reconstruction-oriented representations по action recovery, success/fail separation, planning и OOD robustness. В одном policy-in-the-loop сравнении consensus planning с V-JEPA 2.1 получил success `0.344`, с Cosmos representation — `0.244`. Для нас это аргумент в пользу semantic future error.

### DreamDojo

Источники: [локальный PDF](articles/2602.06949v1.pdf), [arXiv](https://arxiv.org/abs/2602.06949).

DreamDojo использует ensemble policy checkpoints, video world model и внешний video value model для выбора действия. На задачах с высокой вариативностью planning почти удвоил результат относительно равномерного выбора и дал около `+17%` относительно лучшего отдельного checkpoint. Авторы также отмечают систематическое завышение predicted success и трудность тонких физических ошибок, что мотивирует отдельную калибровку риска.

### LIBERO-Safety

Источники: [локальный PDF](articles/2606.23686v2.pdf), [arXiv](https://arxiv.org/abs/2606.23686).

LIBERO-Safety показывает деградацию сильных VLA на physical/semantic safety и OOD. Ошибки включают collision, неудачную траекторию, timeout, кинематический тупик и неверную интерпретацию команды. Это более содержательный benchmark uncertainty-aware planning, чем saturated standard LIBERO.

### pi0.5, pi*0.6 / RECAP и mimic-video

Локальные материалы: [pi0.5](articles/2504.16054v1.pdf), [pi*0.6 / RECAP](articles/2511.14759v2.pdf), [mimic-video](articles/2512.15692v2.pdf).

Эти работы задают сильные ориентиры по open-world generalization, rollout feedback и video-latent planning, но сами по себе не дают калиброванного test-time uncertainty score для нескольких action chunks.

## 15. VFD / SAVE: ключевая статья для epistemic uncertainty

Источники: [локальный PDF](articles/2606.18043v1.pdf), [arXiv](https://arxiv.org/abs/2606.18043), [project page](https://tum-lsy.github.io/uq_vla/), [код](https://github.com/learnsyslab/uq_vla).

Работа **Uncertainty Quantification for Flow-Based Vision-Language-Action Models** оценивает epistemic uncertainty через disagreement velocity fields нескольких независимо fine-tuned flow policies.

Для paper-time $s$, где $s\to1$ соответствует data endpoint:

$$
U_{VFD}(y)=
\frac{1}{M(M-1)N_s}
\sum_{i\ne j}\sum_l
\frac{s_l}{1-s_l}
\|v_{\theta_i}(x^{(i)}_{s_l},y,s_l)
-v_{\theta_j}(x^{(i)}_{s_l},y,s_l)\|_2^2.
$$

Критическая деталь — cross-evaluation: обе модели сравниваются в точке одной trajectory. Несколько outputs одного checkpoint не эквивалентны этому estimator.

Теоретическая связь с mutual-information uncertainty получена для OT Gaussian flow paths и зависит от model diversity и корректности flow approximation. Это не универсальная гарантия calibration на робототехнических OOD.

На LIBERO авторы сообщают модуль отрицательной Spearman correlation между uncertainty задачи и success rate около `0.71 ± 0.03`. VFD также используется в SAVE для active data collection:

$$
U_k=\frac1L\sum_l U_{VFD}(o_{k,l}),
\qquad
p(k)=\frac{U_k^\tau}{\sum_{k'}U_{k'}^\tau},
$$

после чего внутри выбранной задачи запрашивается demonstration для наиболее uncertain initial state. Для нашего проекта SAVE имеет смысл после E1: сначала нужно доказать, что Cosmos VFD связан с ошибкой или fail.

## 16. Failure detection и test-time reranking

### Foresight

[Foresight](https://arxiv.org/abs/2606.23085) использует action-conditioned V-JEPA world model. Causal Transformer получает историю latents текущего observation и predicted future после action chunk, затем выдаёт temporal failure score. Functional conformal prediction задаёт адаптивный threshold. Авторы сообщают balanced accuracy около `0.94` на LIBERO-Long и ROC-AUC до `0.93` в отдельных real-robot settings. Это наиболее близкий опубликованный шаблон для нашей задачи раннего предупреждения, но цифры нельзя напрямую сравнивать с нашими небольшими paired rollout.

### TACO

[TACO](https://arxiv.org/abs/2512.02834) выбирает candidate с наибольшей support/pseudo-count оценкой среди успешных training actions:

$$
a^*=\arg\max_{a_i}\operatorname{PseudoCount}(s,a_i).
$$

На LIBERO-Long сообщается улучшение pi0.5 с `94.8%` до `96.6%`. Это сильный practical baseline для E4, хотя support score может отвергать корректные OOD recovery actions.

### ReconVLA

[ReconVLA](https://arxiv.org/abs/2604.16677) оценивает conformal upper bound ошибки candidate относительно expert action:

$$
a^*=\arg\min_{a_i}\widehat Q_{1-\alpha}
[\|a_i-a_{expert}\|\mid s].
$$

Преимущество — калиброванная action-error оценка; ограничение — необходимость representative expert/calibration data.

### FIPER, ActProbe и SCALE

[FIPER](https://arxiv.org/abs/2510.09459) объединяет observation novelty и action-chunk entropy с conformal calibration на successful rollout. [ActProbe](https://arxiv.org/abs/2606.08508) использует temporal inconsistency перекрывающихся chunks и action magnitude. [SCALE](https://arxiv.org/abs/2602.04208) извлекает single-pass internal uncertainty. Эти методы важны как дешёвые baselines к VFD, который требует минимум две модели и дополнительные denoiser evaluations.

## 17. Temporal overlap consistency: новая проверяемая гипотеза

### Постановка

Cosmos предсказывает chunk длины $H=16$, но контроллер выполняет только $K=8$ действий. Старый неисполненный tail и новый prefix после свежего реального observation относятся к одним absolute simulator steps:

$$
X_q=A_q[8:16],\qquad Y_{q+1}=A_{q+1}[0:8].
$$

Если execution развивается ожидаемо, эти планы должны быть согласованы. Если камера после первых восьми действий обнаружила slip, missed grasp, contact loss или несовместимую смену mode, disagreement может вырасти **до исполнения** нового prefix. Это сигнал revision/surprise, а не гарантия fail: большой score может означать полезную коррекцию, а низкий — уверенно неправильный plan.

Простейший selected-plan baseline:

$$
D_q^{TIDE}=\frac{1}{8D}\sum_{l=0}^{7}\sum_{d=1}^{D}
\left(\frac{X_q[l,d]-Y_{q+1}[l,d]}{s_d}\right)^2,
$$

где $s_d$ — robust scale каждой action dimension по successful calibration rollout. Для stochastic best-of-$N$ дополнительно сравниваются два множества chunks через MMD/energy/Chamfer, а selected old tail — с ближайшим новым candidate support.

### Что уже сделано в статьях

- [Sentinel / STAC](https://proceedings.mlr.press/v270/agia25a.html) использует ровно distributional old-tail/new-prefix comparison; для PushT — $H=16,K=8$. Это прямой baseline, а не наша новая метрика.
- [Rewind-IL / TIDE](https://arxiv.org/html/2604.16683) использует aligned inter-chunk MSE, split conformal threshold на successes и recovery. Авторы проверяют ACT, flow matching, RoboCasa и real robot.
- [BID](https://arxiv.org/abs/2408.17355) штрафует backward overlap inconsistency при выборе нового candidate. Важное ограничение: старый plan может быть неверен после неожиданной динамики.
- [SEAM](https://arxiv.org/abs/2607.04609) направляет flow generation старым unexecuted tail и проверяется на LIBERO-10; слишком сильная consistency ухудшает success.
- [Hide-and-Seek](https://arxiv.org/html/2605.30834) на LIBERO-10 обучает LSTM по frozen action embeddings и coarse success/fail labels. Это обязательный supervised upper baseline к training-free STAC/TIDE.
- [VLA-Corrector](https://arxiv.org/abs/2607.01804) сравнивает predicted и actual visual latent dynamics и адаптивно сокращает horizon. Это ближайший вариант для image/state, но он измеряет ошибку после выполненного prefix.

### План проверки

1. Сохранять все action chunks формы $[Q,N,16,7]$, sample seeds, selected index, фактически выполненные actions и локальные timestamps событий. Старые traces с одними `candidate_first_actions` для этого недостаточны.
2. Проверить индексы на synthetic chunks, fixed-seed reproducibility и pure sampling noise floor.
3. Провести passive rollout без изменения planner: standard LIBERO для false-alarm calibration, mixed success/fail LIBERO-PRO и official LIBERO-Safety events для проверки.
4. Сравнить TIDE MSE, STAC MMD, candidate support, coupled-seed distance, текущие action/value uncertainty и их комбинации.
5. Label каждого query: произойдёт ли первый physical event в следующие $R$ steps, $R\in\{8,16,32\}$. Не считать timeout автоматически физической ошибкой и исключать запросы после необратимого event.
6. Основные метрики: event AUPRC, TPR при frozen FPR 5%, median lead time, Brier/calibration и cluster-bootstrap CI по episodes/cases. Split выполняется по целым tasks/cases.
7. Сравнить global split-CP threshold, phase-conditioned functional CP и Hide-and-Seek-style learned sequential detector.
8. Только после frozen detection test провести paired closed-loop ablation: `max(value)`, weak overlap reranking, alarm-triggered horizon 4 и combined strategy.

Causal planning score для мягкого варианта:

$$
S_i=z(V_i)-\lambda_u z(U_i)-\lambda_o z(D_{overlap,i}),
\qquad i^*=\arg\max_i S_i.
$$

Главный критерий научного результата — положительный lead time на held-out OOD cases и paired рост реального success/safety, а не только корреляция после fail. Полный frozen protocol: [experiments/TEMPORAL_OVERLAP_CONSISTENCY_PROTOCOL_20260820.md](experiments/TEMPORAL_OVERLAP_CONSISTENCY_PROTOCOL_20260820.md).

## 18. Позиционирование и рекомендуемый итоговый метод

Ни одна рассмотренная работа не решает полностью нашу постановку. Наиболее обоснованная система объединяет:

1. Cosmos joint action/future/value generation;
2. VFD независимых LoRA members для epistemic disagreement;
3. semantic DINO/V-JEPA disagreement будущего;
4. cross-query TIDE/STAC overlap disagreement;
5. causal temporal failure predictor по истории queries;
6. conformal reject/replan threshold;
7. counterfactual branch evaluation для обучения candidate ranker.

Для candidate $i$:

$$
\mu_{V,i}=\frac{1}{MN}\sum_{m=1}^{M}\sum_{n=1}^{N}
V_m(\hat s'_{i,m,n}),
$$

$$
R_i=P_\psi(\mathrm{fail}\mid h_{0:t},s_t,a_i,\hat s'_i),
$$

$$
\boxed{
S_i=z(\mu_{V,i})
-\lambda_1z(U_{VFD,i})
-\lambda_2z(U_{semantic,i})
-\lambda_3z(D_{overlap,i})
-\lambda_4z(R_i)
-\lambda_5z(U_{OOD,i})
}
$$

$$
i^*=\arg\max_i S_i.
$$

На текущем этапе нельзя сразу оптимизировать все коэффициенты: признаки коррелированы, а данных мало. Правильная последовательность ablation:

`max(value) -> + TIDE/STAC overlap -> + action VFD -> + future VFD -> + temporal risk -> conformal gate`.

Итоговая научная гипотеза:

> Block-wise epistemic disagreement в joint action–future–value latent sequence предсказывает counterfactual execution error и позволяет выбирать action chunk надёжнее, чем `max(value)` и output-diversity proxies одного checkpoint.

## 19. Краткая позиция относительно SOTA

- **Failure monitoring:** наиболее близкий end-to-end ориентир — Foresight.
- **Epistemic uncertainty flow-policy:** наиболее принципиально корректная идея — VFD/SAVE.
- **Test-time action selection:** сильные практические ориентиры — TACO и ReconVLA.
- **Cross-query consistency:** training-free ориентиры — Sentinel/STAC и Rewind-IL/TIDE; supervised LIBERO baseline — Hide-and-Seek.
- **Дешёвые online signals:** FIPER, ActProbe и SCALE.
- **World-model representation:** semantic V-JEPA/DINO features перспективнее одной reconstruction error.
- **Evaluation:** LIBERO-PRO и LIBERO-Safety полезнее стандартного LIBERO для анализа fail.

Следовательно, наша текущая работа — не готовый SOTA planner, а воспроизводимый переход от heuristic uncertainty к строгому candidate-conditioned epistemic risk. Ближайший необходимый шаг — независимый LoRA ensemble и counterfactual E4 на одинаковых simulator states.